# Tutorial: Calculate cohort outcomes and MT admissions by LSOA

Previously we have calculated the geography and basic outcomes by LSOA, so we have assigned stroke units and travel times, calculated treatment times, and calculated patient outcomes at those treatment times.

Here, we will combine the separate outcomes into the final cohort. We will use the rate of treatment with MT, the proportion of patients who have an LVO, and the proportion of patients who receive a transfer for MT. This will let us know the cohort outcomes and numbers of admissions for thrombectomy.

We will look at the base scenario and recreate the data in these output files:
+ `results_full_base.csv`
+ `results_stats_base.csv`
+ `results_stats_by_unit_base.csv`

__Aim: recreate the cohort outcomes and MT admissions__

In the main run, the steps are combined in a single function, `calculate_cohort_outcomes_admissions`. In this notebook we will break this function down into steps.

The main methods are:
1. calculate admissions for MT and numbers of transfers
2. calculate net benefit of treatment vs no treatment
3. calculate admissions-weighted outcomes

The code in this function is as follows:

In [1]:
# Add path to "classes" directory:
import sys
sys.path.append('../')

import inspect

from classes.add_net_benefit import calculate_cohort_outcomes_admissions

In [2]:
print(inspect.getsource(calculate_cohort_outcomes_admissions))

def calculate_cohort_outcomes_admissions(
        df_geo_outcomes,
        options_prop_lvo_dripship_treated_with_mt,
        col_admissions,
        mt_rate,
        prop_lvo,
    ):
    results = df_geo_outcomes.copy()
    
    list_of_dfs = [results]
    for p in options_prop_lvo_dripship_treated_with_mt:    
        # Don't overwrite shared results:
        results_here = df_geo_outcomes.copy()
        
        # Calculate numbers of thrombectomies and transfers:
        results_here = add_mt_and_transfers(
            results_here, col_admissions, mt_rate, prop_lvo_ds_mt=p)
        # Calculate outcome results that depend on transfers:
        # Add net benefit columns
        results_here = add_net_benefit(
            results_here,
            prop_lvo=prop_lvo,
            prop_lvo_ds_mt=p
        )
        # Calculate admissions-weighted outcomes:
        cols_outcome = ['drip_ship_mrs_0-2', 'lvo_drip_ship_ivt_mt_mrs_0-2']
        if p != 1.0:
            cols_outcome = [f'{c}_

## Code setup

In [3]:
import os
import numpy as np
import pandas as pd

from classes.add_net_benefit import add_mt_and_transfers, add_net_benefit, \
    add_admissions_weighted_outcomes

In [4]:
dir_output = os.path.join('..', 'output')
file_geo_outcomes = 'results_geo_outcomes_base.csv'

## Load data

The geography and basic outcomes data for the base scenario:

In [5]:
p = os.path.join(dir_output, file_geo_outcomes)
df_geo_outcomes = pd.read_csv(p, index_col='LSOA')
cols_start = df_geo_outcomes.columns

/tmp/ipykernel_39534/1643180273.py:2: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  df_geo_outcomes = pd.read_csv(p, index_col='LSOA')


In [6]:
display(df_geo_outcomes.head(3))

,nearest_ivt_unit,nearest_ivt_adjusted_time,nearest_ivt_unit_multiple_same_time,nearest_ivt_unit_multiple_same_time_and_distance,nearest_ivt_unit_multiple_same_time_and_distance_options,nearest_ivt_time,nearest_ivt_unit_no_bias,nearest_ivt_time_no_bias,nearest_mt_time,nearest_mt_unit,...,lvo_mothership_ivt_utility,lvo_mothership_ivt_utility_shift,lvo_mothership_mt_mrs_0-2,lvo_mothership_mt_mrs_shift,lvo_mothership_mt_utility,lvo_mothership_mt_utility_shift,lvo_mothership_ivt_mt_mrs_0-2,lvo_mothership_ivt_mt_mrs_shift,lvo_mothership_ivt_mt_utility,lvo_mothership_ivt_mt_utility_shift
LSOA,,,,,,,,,,,,,,,,,,,,,
ADUR001A,BN25BE,10.9,NaN,NaN,NaN,11,BN25BE,11,11,BN25BE,...,0.38762,0.05501,0.429,-0.785,0.48819,0.15558,0.429,-0.785,0.48819,0.15558
ADUR001B,BN25BE,11.9,NaN,NaN,NaN,12,BN112DH,12,12,BN25BE,...,0.38734,0.05473,0.429,-0.782,0.48760,0.15499,0.429,-0.782,0.48760,0.15499
ADUR001C,BN112DH,11.0,NaN,NaN,NaN,11,BN112DH,11,13,BN25BE,...,0.38706,0.05445,0.428,-0.779,0.48700,0.15439,0.428,-0.779,0.48700,0.15439


In [7]:
print(cols_start)

Index(['nearest_ivt_unit', 'nearest_ivt_adjusted_time',
       'nearest_ivt_unit_multiple_same_time',
       'nearest_ivt_unit_multiple_same_time_and_distance',
       'nearest_ivt_unit_multiple_same_time_and_distance_options',
       'nearest_ivt_time', 'nearest_ivt_unit_no_bias',
       'nearest_ivt_time_no_bias', 'nearest_mt_time', 'nearest_mt_unit',
       'nearest_mt_unit_multiple_same_time',
       'nearest_mt_unit_multiple_same_time_and_distance',
       'nearest_mt_unit_multiple_same_time_and_distance_options',
       'transfer_time', 'transfer_unit', 'transfer_unit_multiple_same_time',
       'transfer_unit_multiple_same_time_and_distance',
       'transfer_unit_multiple_same_time_and_distance_options',
       'transfer_required', 'area_lower', 'Admissions', 'England',
       'Admissions_2025_Est', 'drip_ship_ivt_time', 'drip_ship_mt_time',
       'drip_ship_mt_travel_time', 'drip_ship_mt_travel_time_with_delay',
       'nlvo_no_treatment_mrs_0-2', 'nlvo_no_treatment_utility',

## Parameters

Fixed proportion of patients have LVO (rest have nLVO), and options for proportion transferred for MT are gathered in the same results file.

In [8]:
prop_lvo = 0.3
options_prop_lvo_dripship_treated_with_mt = [1.0, 0.67]

# MT rate only affects admissions and transfers, not outcomes:
mt_rate = 0.1

col_admissions = 'Admissions_2025_Est'

For this example, use this proportion of patients who receive a transfer:

In [9]:
prop_lvo_ds_mt = options_prop_lvo_dripship_treated_with_mt[-1]

Place results in this DataFrame:

In [10]:
# Don't overwrite shared results:
results_here = df_geo_outcomes.copy()

## 1. Calculate MT admissions

How many patients receive MT? The maximum possible amount is the MT rate of 10% of all patients. Then we can check to see whether a transfer is required, and if so reduce the number of patients who receive a transfer.

Place results in these new column names and this new DataFrame:

In [11]:
s = '' if prop_lvo_ds_mt == 1.0 else f'_transferloss'
col_new_ad = f'admissions_mt{s}'
col_new_tr = f'transfers_mt{s}'

df = results_here.copy()

Use the "transfer_required" flag from the geography data to decide whether patients need a transfer for MT.

In [12]:
m = df['transfer_required']

Calculate the MT rate assuming that 10% of all admissions receive MT. Then, for LSOA where a transfer is required, multiply the result by the lossy transfer rate.

In [13]:
# Start with everybody receiving MT:
df[col_new_ad] = df[col_admissions] * mt_rate
# Account for drip-and-ship transfer loss:
if prop_lvo_ds_mt != 1.0:
    df.loc[m, col_new_ad] *= prop_lvo_ds_mt

In [14]:
display(df[['transfer_required', col_admissions, col_new_ad]].head())

,transfer_required,Admissions_2025_Est,admissions_mt_transferloss
LSOA,,,
ADUR001A,False,1.49403,0.149403
ADUR001B,False,1.86762,0.186762
ADUR001C,True,2.24110,0.150154
ADUR001D,False,2.24110,0.224110
ADUR001E,True,4.10862,0.275278


Where transfer required is False, the admissions for MT are 10% of the original admissions. When a transfer is required, the admissions for MT are 10% * 67% of the original admissions.

Separately, keep track of transfers. These values are the same as admissions for MT, unless the LSOA's first unit offers MT, in which case there are zero transfers.

In [15]:
# If the first unit offers MT, set transfers to zero:
df[col_new_tr] = df[col_new_ad].copy()
df.loc[~m, col_new_tr] = 0.0

In [16]:
display(df[['transfer_required', col_admissions, col_new_tr]].head())

,transfer_required,Admissions_2025_Est,transfers_mt_transferloss
LSOA,,,
ADUR001A,False,1.49403,0.000000
ADUR001B,False,1.86762,0.000000
ADUR001C,True,2.24110,0.150154
ADUR001D,False,2.24110,0.000000
ADUR001E,True,4.10862,0.275278


Now the transfers column is only non-zero when the transfer is required.

Round the results:

In [17]:
df[[col_new_ad, col_new_tr]] = df[[col_new_ad, col_new_tr]].round(3)

Check that this does the same as the function:

In [18]:
# Calculate numbers of thrombectomies and transfers:
results_here = add_mt_and_transfers(
    results_here, col_admissions, mt_rate, prop_lvo_ds_mt=prop_lvo_ds_mt)

The following cell will print "True" if these methods are equivalent:

In [19]:
df.equals(results_here)

True

## 2. Net benefit

Net benefit depends on the number of transfers, so calculate it next. Use the same dataframe as the previous section.

First set up some parameters:

In [20]:
cols_start = df.columns
prop_nlvo = 1 - prop_lvo
outcome_columns = ['utility_shift', 'mrs_0-2',
                   'mrs_0-2_improvement_over_no_treatment']

Use the "transfer_required" flag from the geography data to decide whether patients need a transfer for MT.

In [21]:
m_ds = df['transfer_required'] == True

### No treatment

Calculate the cohort results for no treatment:

In [22]:
df['no_treatment_mrs_0-2'] = (
    (df['nlvo_no_treatment_mrs_0-2'] * prop_nlvo) +
    (df['lvo_no_treatment_mrs_0-2'] * prop_lvo)
)

### Cohort IVT & MT

Calculate the cohort results for treatment with IVT and MT.

For the standard drip-and-ship and mothership results, combine the nLVO and LVO outcomes in the ratio of the proportion of patients who have nLVO and LVO.

For the lossy transfer case, let the patients who go directly to an MT unit keep these results. For patients who need a transfer, recalculate the outcomes with a combination of treated nLVO patients, LVO patients treated with both IVT and MT (received transfer), and LVO patients treated with only IVT (did not receive a transfer).

In [23]:
# Utility and proportions mRS 0-2 for each scenario
for s in ['utility_shift', 'mrs_0-2']:
    # Assume everyone can have MT:
    n = f'drip_ship_{s}'
    df[n] = (
        (df[f'nlvo_drip_ship_ivt_{s}'] * prop_nlvo) +
        (df[f'lvo_drip_ship_ivt_mt_{s}'] * prop_lvo)
    )
    if prop_lvo_ds_mt != 1.0:
        # Scale for patients who don't get a transfer for MT:
        n = f'drip_ship_{s}_transferloss'
        df[n] = df[f'drip_ship_{s}'].copy()
        df.loc[m_ds, n] = (
            (df.loc[m_ds, f'nlvo_drip_ship_ivt_{s}'] * prop_nlvo) +
            (df.loc[m_ds, f'lvo_drip_ship_ivt_mt_{s}'] *
             prop_lvo * prop_lvo_ds_mt) +
            (df.loc[m_ds, f'lvo_drip_ship_ivt_{s}'] *
             prop_lvo * (1.0 - prop_lvo_ds_mt))
        )

    df[f'mothership_{s}'] = (
        (df[f'nlvo_mothership_ivt_{s}'] * prop_nlvo) +
        (df[f'lvo_mothership_ivt_mt_{s}'] * prop_lvo)
    )

Check results so far:

In [24]:
cols_cohort = list(set(df.columns) - set(cols_start))

print(cols_cohort)

['no_treatment_mrs_0-2', 'drip_ship_utility_shift', 'drip_ship_utility_shift_transferloss', 'drip_ship_mrs_0-2', 'mothership_mrs_0-2', 'drip_ship_mrs_0-2_transferloss', 'mothership_utility_shift']


In [25]:
cols_to_show = ['transfer_required', 'drip_ship_mrs_0-2',
                'drip_ship_mrs_0-2_transferloss']
cols_to_show = [c for c in cols_to_show if c in df.columns]

display(df[cols_to_show].head(5))

,transfer_required,drip_ship_mrs_0-2,drip_ship_mrs_0-2_transferloss
LSOA,,,
ADUR001A,False,0.6187,0.618700
ADUR001B,False,0.6187,0.618700
ADUR001C,True,0.6028,0.597751
ADUR001D,False,0.6187,0.618700
ADUR001E,True,0.6038,0.598652


The proportion of patients with mRS 0-2 in the transfer loss case (`drip_ship_mrs_0-2_transferloss`) is lower than the base case (`drip_ship_mrs_0-2`) when `transfer_required` is True. Otherwise, the values are the same.

### Improvement over no treatment

The following lines calculate the difference between each cohort's treated and no-treatment outcomes.

In [26]:
# Get mRS 0-2 improvements
# Whole cohort:
for s in ['drip_ship', 'mothership']:
    df[f'{s}_mrs_0-2_improvement_over_no_treatment'] = \
        df[f'{s}_mrs_0-2'] - df['no_treatment_mrs_0-2']
if prop_lvo_ds_mt != 1.0:
    df[f'drip_ship_mrs_0-2_improvement_over_no_treatment_transferloss'] = (
        df[f'drip_ship_mrs_0-2_transferloss'] - df['no_treatment_mrs_0-2'])

# By occlusion-treatment pair:
for (o, t) in [('nlvo', 'ivt'), ('lvo', 'ivt'), ('lvo', 'ivt_mt')]:
    n = f'{o}_drip_ship_{t}_mrs_0-2_improvement_over_no_treatment'
    df[n] = (
        df[f'{o}_drip_ship_{t}_mrs_0-2'] -
        df[f'{o}_no_treatment_mrs_0-2']
    )

Check new columns:

In [27]:
cols_improve = list(set(df.columns) - (set(cols_cohort) | set(cols_start)))

print(cols_improve)

['drip_ship_mrs_0-2_improvement_over_no_treatment_transferloss', 'nlvo_drip_ship_ivt_mrs_0-2_improvement_over_no_treatment', 'drip_ship_mrs_0-2_improvement_over_no_treatment', 'lvo_drip_ship_ivt_mrs_0-2_improvement_over_no_treatment', 'mothership_mrs_0-2_improvement_over_no_treatment', 'lvo_drip_ship_ivt_mt_mrs_0-2_improvement_over_no_treatment']


### Lossy LVO outcomes

Calculate new outcomes for a mixed LVO cohort where some patients receive only IVT and others receive both IVT and MT.

In [28]:
if prop_lvo_ds_mt != 1.0:
    # Calculate results for mixed LVO cohort - some transferred
    # and receive MT, some only receive IVT.
    for s in ['mrs_0-2', 'mrs_0-2_improvement_over_no_treatment']:  # outcome_columns:
        # Start with all IVT+MT results, then for patients who need
        # a transfer, use a mix of IVT-only and IVT+MT.
        n = f'lvo_drip_ship_ivt_mt_{s}_transferloss'
        df[n] = df[f'lvo_drip_ship_ivt_mt_{s}'].copy()
        df.loc[m_ds, n] = (
            (df.loc[m_ds, f'lvo_drip_ship_ivt_mt_{s}'] * prop_lvo_ds_mt) +
            (df.loc[m_ds, f'lvo_drip_ship_ivt_{s}'] * (1.0 - prop_lvo_ds_mt))
        )

In [29]:
cols_lvo = list(set(df.columns) -
                (set(cols_improve) | set(cols_cohort) | set(cols_start)))

print(cols_lvo)

['lvo_drip_ship_ivt_mt_mrs_0-2_transferloss', 'lvo_drip_ship_ivt_mt_mrs_0-2_improvement_over_no_treatment_transferloss']


### Check results

First round our new data:

In [30]:
cols_new = list(set(df.columns) - set(cols_start))
df[cols_new] = df[cols_new].round(5)

Run the original function:

In [31]:
# Calculate outcome results that depend on transfers:
# Add net benefit columns
results_here = add_net_benefit(
    results_here,
    prop_lvo=prop_lvo,
    prop_lvo_ds_mt=prop_lvo_ds_mt
)

The following cell will print "True" if these methods are equivalent:

In [32]:
df.equals(results_here)

True

## 3. Admissions-weighted outcomes

These will be useful later when aggregating outcomes by unit.

Calculate admissions-weighted versions of these columns:

In [33]:
cols_outcome = ['drip_ship_mrs_0-2', 'lvo_drip_ship_ivt_mt_mrs_0-2']
if prop_lvo_ds_mt != 1.0:
    cols_outcome = [f'{c}_transferloss' for c in cols_outcome]

For each of these columns, multiply the outcomes by the admissions. Then place the new columns into the same dataframe.

In [34]:
list_weighted_outcomes = []
for col in cols_outcome:
    s = df[col] * df[col_admissions]
    s = s.rename(f'admissions_weighted_{col}')
    list_weighted_outcomes.append(s)

df = pd.concat([df] + list_weighted_outcomes, axis='columns')
df = df.round(5)

View the new columns:

In [35]:
for col in cols_outcome:
    display(df[[col_admissions, col, f'admissions_weighted_{col}']].head(3)) 

,Admissions_2025_Est,drip_ship_mrs_0-2_transferloss,admissions_weighted_drip_ship_mrs_0-2_transferloss
LSOA,,,
ADUR001A,1.49403,0.61870,0.92436
ADUR001B,1.86762,0.61870,1.15550
ADUR001C,2.24110,0.59775,1.33962


,Admissions_2025_Est,lvo_drip_ship_ivt_mt_mrs_0-2_transferloss,admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss
LSOA,,,
ADUR001A,1.49403,0.42900,0.64094
ADUR001B,1.86762,0.42900,0.80121
ADUR001C,2.24110,0.35917,0.80494


### Check results

Run the original function:

In [36]:
results_here = add_admissions_weighted_outcomes(
    results_here, col_admissions, cols_outcome)

The following cell will print "True" if these methods are equivalent:

In [37]:
df.equals(results_here)

True

## Tidying up

The rest of `calculate_cohort_outcomes_admissions` gathers the results for multiple options for the transfer rate, rounds the new data, and renames columns from `transferloss` to include the transfer loss proportion. For example, for a 67% transfer rate, the columns are labelled `transferloss0!67`. The exclamation mark is used instead of a full stop to prevent any confusion of accessing column names that contain a `.`.

## Conclusion

This notebook shows how cohort results are calculated at the LSOA level. We calculated the numbers of thrombectomy and transfers, the benefit of this scenario over no treatment, and the admissions-weighted outcomes.